In [1]:
import chromadb
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [2]:
df = pd.read_parquet("../data/processed/arxiv_ml.parquet")

df.head()

,id,title,authors,category,text_raw,text_ml
0,704.0001,Calculation of prompt diphoton production cros...,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",hep-ph,Calculation of prompt diphoton production cros...,calculation prompt diphoton production cross s...
1,704.0002,Sparsity-certifying Graph Decompositions,Ileana Streinu and Louis Theran,math.CO,Sparsity-certifying Graph Decompositions We ...,sparsity certify graph decomposition describe ...
2,704.0003,The evolution of the Earth-Moon system based o...,Hongjun Pan,physics.gen-ph,The evolution of the Earth-Moon system based o...,evolution earth moon system base dark matter f...
3,704.0004,A determinant of Stirling cycle numbers counts...,David Callan,math.CO,A determinant of Stirling cycle numbers counts...,determinant stirling cycle number count unlabe...
4,704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,Wael Abu-Shammala and Alberto Torchinsky,math.CA,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,dyadic lambda alpha lambda alpha compute lambd...


In [3]:
print(df.shape)
print(df.columns)

(1000, 6)
Index(['id', 'title', 'authors', 'category', 'text_raw', 'text_ml'], dtype='str')


In [4]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
embeddings = embedding_model.encode(
    df["text_raw"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [6]:
print(embeddings.shape)

(1000, 384)


In [7]:
embeddings = embedding_model.encode(
    df["text_raw"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(1000, 384)


In [8]:
import os

os.makedirs("../data/embeddings", exist_ok=True)

np.save(
    "../data/embeddings/embeddings.npy",
    embeddings
)

print("Embeddings saved successfully.")

Embeddings saved successfully.


In [9]:
loaded_embeddings = np.load(
    "../data/embeddings/embeddings.npy"
)

print(loaded_embeddings.shape)

(1000, 384)


In [10]:
import chromadb

client = chromadb.PersistentClient(path="../chroma_db")

print("ChromaDB Connected Successfully!")

ChromaDB Connected Successfully!


In [11]:
try:
    client.delete_collection("research_papers")
    print("Old collection deleted.")
except:
    print("No existing collection found.")

No existing collection found.


In [12]:
collection = client.create_collection(
    name="research_papers"
)

print(collection)

Collection(name=research_papers)


In [13]:
ids = df["id"].astype(str).tolist()

documents = df["text_raw"].tolist()

metadatas = [
    {
        "title": row.title,
        "authors": row.authors,
        "category": row.category
    }
    for _, row in df.iterrows()
]

embedding_list = embeddings.tolist()

In [14]:
print(len(ids))
print(len(documents))
print(len(metadatas))
print(len(embedding_list))

1000
1000
1000
1000


In [15]:
collection.add(
    ids=ids,
    embeddings=embedding_list,
    documents=documents,
    metadatas=metadatas
)

In [16]:
print("Total Documents :", collection.count())

Total Documents : 1000


In [17]:
query = "Black Holes"

query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True
)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
    include=["metadatas", "distances"]
)

In [18]:
for i, meta in enumerate(results["metadatas"][0], start=1):

    print("=" * 80)
    print(f"Paper {i}")
    print("=" * 80)

    print("Title    :", meta["title"])
    print("Authors  :", meta["authors"])
    print("Category :", meta["category"])
    print("Distance :", round(results["distances"][0][i-1], 4))
    print()

Paper 1
Title    : Topology Change of Black Holes
Authors  : Daisuke Ida and Masaru Siino
Category : gr-qc
Distance : 0.8287

Paper 2
Title    : The First Law for Boosted Kaluza-Klein Black Holes
Authors  : David Kastor, Sourya Ray and Jennie Traschen
Category : hep-th
Distance : 0.943

Paper 3
Title    : Hawking radiation of linear dilaton black holes
Authors  : G. Clement, J.C. Fabris and G.T. Marques
Category : gr-qc
Distance : 0.9928

Paper 4
Title    : General Relativity Today
Authors  : Thibault Damour
Category : gr-qc
Distance : 1.002

Paper 5
Title    : Binary Systems as Test-beds of Gravity Theories
Authors  : Thibault Damour
Category : gr-qc
Distance : 1.023

